# Module 8: LLMOps & Observability

**Goal:** Instrument every LLM call so you can track cost, quality, and failures before users complain.

**What you'll learn:**
1. The three pillars of LLM observability (traces, metrics, feedback)
2. Structured logging for every LLM call
3. Cost tracking and budget alerting
4. Quality drift detection
5. Prompt versioning and A/B tracking
6. Building a live dashboard summary

---

## 0. Setup

In [ ]:
%pip install -q openai python-dotenv

In [ ]:
import os, json, time, uuid, statistics
from datetime import datetime
from dataclasses import dataclass, asdict, field
from collections import defaultdict
from typing import Optional
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from openai import OpenAI
client = OpenAI()

PRICING = {"gpt-4o-mini": (0.15e-6, 0.60e-6), "gpt-4o": (2.5e-6, 10e-6)}
print("✅ Ready")

---
## 1. The Three Pillars

```
TRACES   — What happened, step by step.
           "User asked X → retrieved chunks A,B → model produced Y in 640ms"

METRICS  — Aggregated numbers over time.
           "p95 latency this hour: 890ms, avg cost today: $0.0042/req"

FEEDBACK — Signal about quality.
           "User thumbed-down 3 responses → all had hallucinated citations"
```

You need all three. Traces alone tell you *what* happened; metrics tell you *trends*; feedback tells you *why* it matters.

---
## 2. Structured Tracing

In [ ]:
@dataclass
class LLMTrace:
    trace_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    model: str = "gpt-4o-mini"
    prompt: str = ""
    response: str = ""
    input_tokens: int = 0
    output_tokens: int = 0
    latency_ms: float = 0.0
    cost_usd: float = 0.0
    error: Optional[str] = None
    metadata: dict = field(default_factory=dict)

    def __post_init__(self):
        ip, op = PRICING.get(self.model, PRICING["gpt-4o-mini"])
        self.cost_usd = self.input_tokens * ip + self.output_tokens * op


# In-memory trace store (use a real DB in production)
trace_store: list[LLMTrace] = []


def traced_call(prompt: str, model="gpt-4o-mini", **metadata) -> LLMTrace:
    """Wrap any LLM call with full tracing."""
    trace = LLMTrace(model=model, prompt=prompt[:200], metadata=metadata)
    start = time.perf_counter()
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        trace.response = resp.choices[0].message.content
        trace.input_tokens = resp.usage.prompt_tokens
        trace.output_tokens = resp.usage.completion_tokens
        trace.__post_init__()  # recalculate cost with real token counts
    except Exception as e:
        trace.error = str(e)
    trace.latency_ms = (time.perf_counter() - start) * 1000
    trace_store.append(trace)
    return trace


# Generate some traces
prompts = [
    "What is the capital of Japan?",
    "Explain gradient descent in one sentence.",
    "Write a haiku about machine learning.",
    "What is 17 * 23?",
    "Name three vector databases.",
]
for p in prompts:
    t = traced_call(p, feature="demo")
    print(f"[{t.trace_id}] {t.latency_ms:5.0f}ms  ${t.cost_usd:.6f}  {t.response[:50]}...")

---
## 3. Metrics Aggregation

In [ ]:
def compute_metrics(traces: list[LLMTrace]) -> dict:
    if not traces:
        return {}

    successful = [t for t in traces if not t.error]
    latencies  = sorted(t.latency_ms for t in successful)
    costs      = [t.cost_usd for t in successful]
    tokens_out = [t.output_tokens for t in successful]

    def pct(data, p):
        idx = max(0, int(len(data) * p / 100) - 1)
        return data[idx] if data else 0

    return {
        "total_requests": len(traces),
        "error_rate":     round(1 - len(successful) / len(traces), 3),
        "latency_avg_ms": round(statistics.mean(latencies), 1),
        "latency_p50_ms": round(pct(latencies, 50), 1),
        "latency_p95_ms": round(pct(latencies, 95), 1),
        "total_cost_usd": round(sum(costs), 6),
        "avg_cost_usd":   round(statistics.mean(costs), 6),
        "total_tokens_out": sum(tokens_out),
        "models_used":    list({t.model for t in traces}),
    }

metrics = compute_metrics(trace_store)
print("Current metrics:")
for k, v in metrics.items():
    print(f"  {k:22s}: {v}")

---
## 4. Cost Tracking & Budget Alerts

In [ ]:
class BudgetTracker:
    def __init__(self, daily_limit_usd: float = 10.0, alert_at_pct: float = 0.8):
        self.daily_limit = daily_limit_usd
        self.alert_threshold = daily_limit_usd * alert_at_pct
        self._spend: list[tuple[str, float]] = []   # (timestamp, cost)

    def record(self, cost_usd: float):
        self._spend.append((datetime.now().isoformat(), cost_usd))
        total = self.today_spend()
        if total >= self.daily_limit:
            print(f"🚨 BUDGET EXCEEDED: ${total:.4f} / ${self.daily_limit}")
        elif total >= self.alert_threshold:
            print(f"⚠️  Budget alert: ${total:.4f} used (>{self.alert_threshold*100/self.daily_limit:.0f}% of ${self.daily_limit} limit)")

    def today_spend(self) -> float:
        today = datetime.now().date().isoformat()
        return sum(c for ts, c in self._spend if ts.startswith(today))

    def projection(self) -> dict:
        """Project monthly cost based on today's spend rate."""
        daily = self.today_spend()
        return {"today_usd": round(daily, 4), "monthly_projection_usd": round(daily * 30, 2)}


budget = BudgetTracker(daily_limit_usd=0.01, alert_at_pct=0.5)  # Low limit to demo alerts
for t in trace_store:
    budget.record(t.cost_usd)

print(f"\nBudget status: {budget.projection()}")

---
## 5. Quality Drift Detection

In [ ]:
import json

def score_response(question: str, answer: str) -> float:
    """LLM-as-judge: returns 1-5 overall quality score."""
    result = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"""Rate this answer 1-5 for quality. Return JSON: {{"score": <1-5>}}
Q: {question}
A: {answer}"""}],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return json.loads(result.choices[0].message.content).get("score", 3)


class DriftDetector:
    """Detect when average quality drops vs. a baseline window."""

    def __init__(self, window: int = 10, drop_threshold: float = 0.5):
        self.window = window
        self.drop_threshold = drop_threshold
        self.scores: list[float] = []

    def add(self, score: float) -> bool:
        """Returns True if drift detected."""
        self.scores.append(score)
        if len(self.scores) < self.window * 2:
            return False
        baseline = statistics.mean(self.scores[:self.window])
        recent   = statistics.mean(self.scores[-self.window:])
        drift = baseline - recent
        if drift >= self.drop_threshold:
            print(f"🚨 QUALITY DRIFT: baseline={baseline:.2f}, recent={recent:.2f}, drop={drift:.2f}")
            return True
        return False


detector = DriftDetector(window=3, drop_threshold=0.8)

# Simulate: good scores first, then degraded scores
simulated_scores = [4.5, 4.2, 4.8, 4.3, 4.6, 3.1, 2.9, 3.0, 2.8, 3.2]
for i, score in enumerate(simulated_scores):
    drift = detector.add(score)
    print(f"  Score {i+1}: {score}  {'← DRIFT DETECTED' if drift else ''}")

---
## 6. Prompt Versioning

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class PromptVersion:
    name: str
    version: str
    template: str
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    avg_score: float = 0.0
    call_count: int = 0


class PromptRegistry:
    def __init__(self):
        self._registry: dict[str, list[PromptVersion]] = defaultdict(list)

    def register(self, pv: PromptVersion):
        self._registry[pv.name].append(pv)

    def get_active(self, name: str) -> Optional[PromptVersion]:
        versions = self._registry.get(name, [])
        return versions[-1] if versions else None

    def record_score(self, name: str, score: float):
        pv = self.get_active(name)
        if pv:
            pv.call_count += 1
            pv.avg_score = ((pv.avg_score * (pv.call_count - 1)) + score) / pv.call_count

    def history(self, name: str) -> list[dict]:
        return [{"version": v.version, "avg_score": round(v.avg_score, 2), "calls": v.call_count}
                for v in self._registry.get(name, [])]


registry = PromptRegistry()

# Register two versions of a summarisation prompt
registry.register(PromptVersion("summarise", "v1", "Summarise this: {text}"))
registry.register(PromptVersion("summarise", "v2", "Summarise in 2 sentences, keep key numbers: {text}"))

# Simulate scores for each version
for score in [3.2, 3.5, 3.1, 3.4]:   # v2 is active; these go to it
    registry.record_score("summarise", score)

print("Prompt version history:")
for entry in registry.history("summarise"):
    print(f"  {entry}")

---
## 7. Dashboard Summary

---
## 🧪 Exercises

1. **Trace Analysis**: Run 100 requests. What's your p50/p95 latency? Cost per request?
2. **Drift Detection**: Change the system prompt. Does response quality change? How to detect automatically?
3. **Dashboard**: Build a simple dashboard showing: requests/min, avg latency, error rate, cost/day.

In [ ]:
def print_dashboard(traces: list[LLMTrace]):
    m = compute_metrics(traces)
    print("\n" + "═" * 55)
    print("  LLMOps Dashboard")
    print("═" * 55)
    print(f"  Requests:       {m['total_requests']}")
    print(f"  Error rate:     {m['error_rate']:.1%}")
    print(f"  Latency avg:    {m['latency_avg_ms']:.0f}ms")
    print(f"  Latency p95:    {m['latency_p95_ms']:.0f}ms")
    print(f"  Total cost:     ${m['total_cost_usd']:.6f}")
    print(f"  Cost / request: ${m['avg_cost_usd']:.6f}")
    print(f"  Monthly proj:   ${m['avg_cost_usd'] * 1_000_000 * 30:,.0f}  (at 1M req/day)")
    print(f"  Models:         {', '.join(m['models_used'])}")
    print("═" * 55)

print_dashboard(trace_store)

---
## 🧪 Exercises

1. **Persist traces**: Write `trace_store` to a `.jsonl` file after every call. Reload it on startup. Now your metrics survive restarts.
2. **Alert on latency spike**: Add a check in `traced_call` that prints a warning if any single call exceeds 3000ms.
3. **Real quality scoring**: Score 10 responses from `trace_store` using the `score_response` function. Which prompts got the lowest scores? Why?
4. **A/B prompt test via registry**: Register two versions of a prompt, run 20 calls split 50/50, score each. Which version wins?
5. **Langfuse integration**: Sign up at [cloud.langfuse.com](https://cloud.langfuse.com) (free), add your keys to `.env`, and replace the in-memory store with `langfuse.trace()` calls.

---
**Next:** [Module 09 — EvalOps](../09-eval-ops/eval_ops.ipynb)